**Load features**

In [1]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import numpy as np

In [2]:
train_features = np.load("features\\train_data.npy")
test_features = np.load("features\\test_data.npy")

train_labels = np.load("features\\train_labels.npy")
test_labels = np.load("features\\test_labels.npy")

# Transform
encoder = OneHotEncoder(sparse_output=False)
train_labels = encoder.fit_transform(train_labels.reshape(-1, 1))
test_labels = encoder.transform(test_labels.reshape(-1, 1))

standardScaler = StandardScaler()
train_features = standardScaler.fit_transform(train_features)
test_features = standardScaler.transform(test_features)

print(train_features.shape)
print(train_labels.shape)
print(test_features.shape)
print(test_labels.shape)

(1152, 162)
(1152, 8)
(288, 162)
(288, 8)


**Upload processed spectrogram images**

In [ ]:
# Load data
train_images       = np.load("features\\train_images.npy")
test_images        = np.load("features\\test_images.npy")
train_image_labels = np.load("features\\train_image_labels.npy")
test_image_labels  = np.load("features\\test_image_labels.npy")

# One-hot encode labels trước
encoder = OneHotEncoder(sparse_output=False)
train_image_labels_encoded = encoder.fit_transform(train_image_labels.reshape(-1, 1))
test_image_labels_encoded  = encoder.transform(test_image_labels.reshape(-1, 1))

print(train_images.shape)
print(test_images.shape)
print(train_image_labels_encoded.shape)
print(test_image_labels_encoded.shape)

(1152, 224, 224)
(288, 224, 224)
(1152, 8)
(288, 8)


**Neural network architecture and training:**

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader, TensorDataset

# ── Chuẩn bị data ─────────────────────────────────────────────

# Ảnh grayscale (n, 224, 224) → RGB (n, 3, 224, 224)
x_trainI_rgb = np.repeat(train_images.astype(np.float32, copy=False)[:, None, :, :], 3, axis=1) / np.float32(255.0)
x_testI_rgb  = np.repeat(test_images.astype(np.float32, copy=False)[:, None, :, :], 3, axis=1) / np.float32(255.0)

# Feature vector (n, 162) → (n, 1, 162)
x_trainT = train_features.reshape(-1, 1, 162).astype(np.float32, copy=False)
x_testT  = test_features.reshape(-1, 1, 162).astype(np.float32, copy=False)

# Tensor
X_train_img  = torch.from_numpy(x_trainI_rgb)
X_test_img   = torch.from_numpy(x_testI_rgb)
X_train_feat = torch.from_numpy(x_trainT)
X_test_feat  = torch.from_numpy(x_testT)
y_train      = torch.from_numpy(train_labels).float()
y_test       = torch.from_numpy(test_labels).float()

train_loader = DataLoader(
    TensorDataset(X_train_img, X_train_feat, y_train),
    batch_size=32, shuffle=True
)
test_loader = DataLoader(
    TensorDataset(X_test_img, X_test_feat, y_test),
    batch_size=32
)

# ── Model ─────────────────────────────────────────────────────

class ResNetCNN1D(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()

        # Branch 1: ResNet18
        base_resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        for param in base_resnet.parameters():
            param.requires_grad = False  # Freeze

        # Lấy backbone đến trước fc layer -> output (n, 512, 1, 1)
        self.resnet_backbone = nn.Sequential(*list(base_resnet.children())[:-1])

        # Branch 2: CNN1D cho feature vector
        self.cnn1d = nn.Sequential(
            nn.Conv1d(1, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.25),

            nn.Conv1d(128, 256, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.45),

            nn.Conv1d(256, 128, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.5),

            nn.Flatten()  # (n, 128 * 20) = (n, 2560)
        )

        # Kết hợp 2 branch
        # ResNet output: 512
        # CNN1D output:  128 * (162 // 8) = 128 * 20 = 2560
        self.classifier = nn.Sequential(
            nn.Linear(512 + 2560, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(128, num_classes),
            nn.Softmax(dim=1)
        )
    def forward(self, img, feat):
        # feat shape: (n, 1, 162)
        x1 = self.resnet_backbone(img)          # (n, 512, 1, 1)
        x1 = torch.flatten(x1, 1)               # (n, 512)
        x2 = self.cnn1d(feat)                   # (n, 2560)

        x = torch.cat([x1, x2], dim=1)          # (n, 3072)
        return self.classifier(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

model = ResNetCNN1D(num_classes=8).to(device)

# ── Train ─────────────────────────────────────────────────────

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=5, factor=0.5
)


def train_epoch():
    model.train()
    total_loss, correct, total_samples = 0, 0, 0
    for img, feat, label in train_loader:
        img, feat, label = img.to(device), feat.to(device), label.to(device)
        optimizer.zero_grad()
        output = model(img, feat)
        loss   = criterion(output, label)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += (output.argmax(1) == label.argmax(1)).sum().item()
        total_samples += label.size(0)
    return total_loss / len(train_loader), correct / total_samples


def eval_epoch():
    model.eval()
    total_loss, correct, total_samples = 0, 0, 0
    with torch.no_grad():
        for img, feat, label in test_loader:
            img, feat, label = img.to(device), feat.to(device), label.to(device)
            output = model(img, feat)
            loss   = criterion(output, label)
            total_loss += loss.item()
            correct    += (output.argmax(1) == label.argmax(1)).sum().item()
            total_samples += label.size(0)
    return total_loss / len(test_loader), correct / total_samples


best_acc = 0
EPOCHS = 50

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch()
    test_loss,  test_acc  = eval_epoch()
    scheduler.step(test_acc)

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), "best_model.pth")

    print(f"Epoch {epoch:02d}/{EPOCHS} "
          f"| Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} "
          f"| Test Loss: {test_loss:.4f} Acc: {test_acc:.4f} "
          f"| Best: {best_acc:.4f}")

Using: cuda
Epoch 01/50 | Train Loss: 2.0407 Acc: 0.1858 | Test Loss: 2.0123 Acc: 0.1979 | Best: 0.1979
Epoch 02/50 | Train Loss: 2.0166 Acc: 0.2309 | Test Loss: 1.9685 Acc: 0.3229 | Best: 0.3229
Epoch 03/50 | Train Loss: 1.9578 Acc: 0.3238 | Test Loss: 1.9081 Acc: 0.3576 | Best: 0.3576
Epoch 04/50 | Train Loss: 1.9212 Acc: 0.3507 | Test Loss: 1.8848 Acc: 0.3924 | Best: 0.3924
Epoch 05/50 | Train Loss: 1.8729 Acc: 0.3993 | Test Loss: 1.8739 Acc: 0.3993 | Best: 0.3993
Epoch 06/50 | Train Loss: 1.8616 Acc: 0.4080 | Test Loss: 1.8902 Acc: 0.3750 | Best: 0.3993
Epoch 07/50 | Train Loss: 1.8453 Acc: 0.4271 | Test Loss: 1.8680 Acc: 0.4062 | Best: 0.4062
Epoch 08/50 | Train Loss: 1.8379 Acc: 0.4427 | Test Loss: 1.8666 Acc: 0.3958 | Best: 0.4062
Epoch 09/50 | Train Loss: 1.8209 Acc: 0.4505 | Test Loss: 1.8472 Acc: 0.4097 | Best: 0.4097
Epoch 10/50 | Train Loss: 1.7904 Acc: 0.4835 | Test Loss: 1.8249 Acc: 0.4410 | Best: 0.4410
Epoch 11/50 | Train Loss: 1.7877 Acc: 0.4896 | Test Loss: 1.8364 Acc